# 02 - Wikipedia Evidence Retrieval (Improved)

In [ ]:
import os, re, json, time, random, requests
import pandas as pd
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS


INPUT_CSV = r"../data/Phi3_Claims_Level_Wikipedia_only.csv"
OUTPUT_CSV = r"../data/Prompt_Phi3_Claims_with_Evidence.csv"

TOP_PAGES = 5
TOP_SENTENCES = 3
CONTEXT_WINDOW = 1

EMBED_MODEL_NAME = "BAAI/bge-base-en-v1.5"

CACHE_DIR = r"../data/wiki_cache"
PAGE_CACHE_PATH = os.path.join(CACHE_DIR, "page_cache.json")
SEARCH_CACHE_PATH = os.path.join(CACHE_DIR, "search_cache.json")

MIN_REQUEST_INTERVAL = 0.75
MAX_RETRIES = 6


In [ ]:
df = pd.read_csv(INPUT_CSV)
print("Rows:", len(df))
print("Columns:", df.columns.tolist())

## 1. Context-aware search-query generation

The baseline searched the complete atomic claim directly. That is often a poor Wikipedia query because:
- claims may be long natural-language statements,
- claim extraction can remove useful context from the original question,
- phrases such as "the answer is" add noise.

This version combines **Question + Atomic Claim**, removes common stop words and boilerplate,
deduplicates keywords, and keeps the query short.

This remains a lightweight rule-based query generator. A Qwen-generated query can later be
tested as an ablation against this version.


In [ ]:
QUERY_BOILERPLATE = {
    "answer", "answers", "claim", "claims",
    "according", "states", "stated", "says", "said",
    "true", "false"
}

def _extract_query_terms(text):
    """Extract useful search terms while preserving their original order."""
    text = str(text)
    text = re.sub(r"[^A-Za-z0-9\s\-']", " ", text)
    tokens = text.split()

    terms = []
    for token in tokens:
        low = token.lower()

        if len(token) <= 2:
            continue
        if low in ENGLISH_STOP_WORDS:
            continue
        if low in QUERY_BOILERPLATE:
            continue

        terms.append(token)

    return terms


def generate_search_query(question, claim, max_terms=12):
    """
    Build a short Wikipedia query from both the original question and atomic claim.

    We put claim terms first because they describe the proposition being verified,
    then add non-duplicate question terms to recover missing context.
    """
    claim_terms = _extract_query_terms(claim)
    question_terms = _extract_query_terms(question)

    combined = claim_terms + question_terms

    seen = set()
    unique_terms = []

    for term in combined:
        key = term.lower()
        if key not in seen:
            seen.add(key)
            unique_terms.append(term)

    query = " ".join(unique_terms[:max_terms]).strip()

    if not query:
        query = str(claim).strip()

    return query


## 2. Wikipedia API helper: throttling, retry, and search cache

In [ ]:
WIKI_API = "https://en.wikipedia.org/w/api.php"

SESSION = requests.Session()
SESSION.headers.update({"User-Agent": "HallucinationDetectionCourseProject/1.0 (academic project)"})

os.makedirs(CACHE_DIR, exist_ok=True)

def _load_json_cache(path):
    if not os.path.exists(path):
        return {}

    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except (json.JSONDecodeError, OSError):
        return {}


def _save_json_cache(cache, path):
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(cache, f, ensure_ascii=False)
    os.replace(tmp, path)


PAGE_CACHE = _load_json_cache(PAGE_CACHE_PATH)
SEARCH_CACHE = _load_json_cache(SEARCH_CACHE_PATH)

_last_request_time = 0.0


def wiki_request(params, max_retries=MAX_RETRIES):
    """Wikipedia GET request with global throttling and 429-aware retry."""
    global _last_request_time

    for attempt in range(max_retries):
        elapsed = time.time() - _last_request_time
        if elapsed < MIN_REQUEST_INTERVAL:
            time.sleep(MIN_REQUEST_INTERVAL - elapsed)

        try:
            r = SESSION.get(WIKI_API, params=params, timeout=30)
            _last_request_time = time.time()

            if r.status_code == 429:
                retry_after = r.headers.get("Retry-After")

                if retry_after:
                    try:
                        wait = float(retry_after)
                    except ValueError:
                        wait = None
                else:
                    wait = None

                if wait is None:
                    wait = min(60.0, (2 ** attempt) + random.uniform(0, 1))

                print(f"Rate limited (429). Waiting {wait:.1f}s before retry...")
                time.sleep(wait)
                continue

            if r.status_code in {500, 502, 503, 504}:
                wait = min(60.0, (2 ** attempt) + random.uniform(0, 1))
                print(f"Wikipedia HTTP {r.status_code}. Retry in {wait:.1f}s...")
                time.sleep(wait)
                continue

            r.raise_for_status()
            return r

        except requests.RequestException as e:
            if attempt == max_retries - 1:
                raise

            wait = min(60.0, (2 ** attempt) + random.uniform(0, 1))
            print(f"Request failed: {e}. Retry in {wait:.1f}s...")
            time.sleep(wait)

    raise RuntimeError("Wikipedia request failed after retries.")


def search_wikipedia(query, limit=TOP_PAGES):
    cache_key = f"{limit}::{query.lower().strip()}"

    if cache_key in SEARCH_CACHE:
        return SEARCH_CACHE[cache_key]

    params = {
        "action": "query",
        "list": "search",
        "srsearch": query,
        "format": "json",
        "srlimit": limit
    }

    r = wiki_request(params)
    data = r.json()

    titles = [
        x["title"]
        for x in data.get("query", {}).get("search", [])
    ]

    SEARCH_CACHE[cache_key] = titles
    _save_json_cache(SEARCH_CACHE, SEARCH_CACHE_PATH)

    return titles


## 3. Batch-fetch Wikipedia page text with persistent cache


In [ ]:
def fetch_wikipedia_pages(titles):
    """
    Return {page_title: page_text}.

    Cached titles are reused locally. All missing titles are fetched in one API request.
    """
    if not titles:
        return {}

    results = {}
    missing = []

    for title in titles:
        if title in PAGE_CACHE:
            results[title] = PAGE_CACHE[title]
        else:
            missing.append(title)

    if missing:
        params = {
            "action": "query",
            "prop": "extracts",
            "explaintext": 1,
            "redirects": 1,
            "titles": "|".join(missing),
            "format": "json"
        }

        r = wiki_request(params)
        pages = r.json().get("query", {}).get("pages", {})

        for page in pages.values():
            canonical_title = page.get("title", "")
            text = page.get("extract", "")

            if canonical_title and text:
                results[canonical_title] = text
                PAGE_CACHE[canonical_title] = text

        redirects = r.json().get("query", {}).get("redirects", [])
        redirect_map = {
            item.get("from"): item.get("to")
            for item in redirects
            if item.get("from") and item.get("to")
        }

        for requested_title in missing:
            canonical = redirect_map.get(requested_title, requested_title)
            if canonical in results:
                results[requested_title] = results[canonical]
                PAGE_CACHE[requested_title] = results[canonical]

        _save_json_cache(PAGE_CACHE, PAGE_CACHE_PATH)

    return {
        title: results.get(title, PAGE_CACHE.get(title, ""))
        for title in titles
        if results.get(title, PAGE_CACHE.get(title, ""))
    }


## 4. Split article text into sentences

In [ ]:
def split_into_sentences(text):
    if not isinstance(text, str):
        return []

    text = re.sub(r"\s+", " ", text).strip()
    if not text:
        return []

    sentences = re.split(r"(?<=[.!?])\s+(?=[A-Z0-9\"'])", text)

    return [s.strip() for s in sentences if len(s.strip()) >= 30]

## 5. Sentence-level semantic retrieval

In [ ]:
embed_model = SentenceTransformer(EMBED_MODEL_NAME)
print("Loaded:", EMBED_MODEL_NAME)

In [ ]:
def retrieve_top_k_sentences(
    claim,
    sentences,
    top_k=TOP_SENTENCES,
    context_window=CONTEXT_WINDOW
):
    if not sentences:
        return []

    claim_emb = embed_model.encode([claim], normalize_embeddings=True)
    sent_embs = embed_model.encode(
        sentences,
        normalize_embeddings=True,
        show_progress_bar=False
    )

    scores = cosine_similarity(claim_emb, sent_embs)[0]
    ranked = scores.argsort()[::-1][:top_k]

    results = []
    for idx in ranked:
        start = max(0, idx - context_window)
        end = min(len(sentences), idx + context_window + 1)

        results.append({
            "sentence_index": int(idx),
            "similarity": float(scores[idx]),
            "matched_sentence": sentences[idx],
            "context": " ".join(sentences[start:end])
        })

    return results

## 6. End-to-end evidence retrieval for one claim


In [ ]:
def retrieve_evidence_for_claim(question, claim):
    query = generate_search_query(question, claim)
    page_titles = search_wikipedia(query, limit=TOP_PAGES)

    candidates = []

    # One batched API request for all uncached candidate pages.
    page_texts = fetch_wikipedia_pages(page_titles)

    for title in page_titles:
        try:
            text = page_texts.get(title, "")
            if not text:
                continue

            sentences = split_into_sentences(text)

            hits = retrieve_top_k_sentences(
                claim,
                sentences,
                top_k=TOP_SENTENCES,
                context_window=CONTEXT_WINDOW
            )

            for hit in hits:
                hit["page_title"] = title
                hit["search_query"] = query
                candidates.append(hit)

        except Exception as e:
            print(f"Failed processing page {title}: {e}")

    candidates = sorted(
        candidates,
        key=lambda x: x["similarity"],
        reverse=True
    )

    return {
        "query": query,
        "pages": page_titles,
        "evidence": candidates[:TOP_SENTENCES]
    }


## 7. Test on one atomic claim first

Always test a single row before running the full dataset.
This lets you inspect whether the generated search query and returned pages are sensible.


In [ ]:
# Test retrieval using the first NON-EMPTY claim
valid_test_rows = df[
    df["Atomic_Claim"].notna()
    & df["Atomic_Claim"].astype(str).str.strip().ne("")
]

if len(valid_test_rows) == 0:
    print("No non-empty claims available for retrieval test.")
else:
    test_row = valid_test_rows.iloc[0]
    test_question = str(test_row["Question"])
    test_claim = str(test_row["Atomic_Claim"]).strip()

    print("QUESTION:")
    print(test_question)

    print("\nCLAIM:")
    print(test_claim)

    result = retrieve_evidence_for_claim(test_question, test_claim)

    print("\nQUERY:")
    print(result["query"])

    print("\nPAGES:")
    for p in result["pages"]:
        print("-", p)

    print("\nTOP EVIDENCE:")
    for i, ev in enumerate(result["evidence"], 1):
        print(f"\nEvidence {i}")
        print("Page:", ev["page_title"])
        print("Similarity:", round(ev["similarity"], 4))
        print("Matched sentence:", ev["matched_sentence"])
        print("Context:", ev["context"])


## 8. Run over the whole claim-level dataset

This saves the top 3 evidence contexts and their similarity scores.

Notes:
- The notebook can resume from an existing output CSV.
- Wikipedia search/page caches persist under `../data/wiki_cache/`.
- Do **not** add an extra `sleep()` per claim: request pacing is already handled centrally by `wiki_request()`.
- A low similarity score means retrieval was weak; it does **not** automatically mean hallucination.


### Empty-claim handling

Rows with an empty `Atomic_Claim` are **kept** in the dataset but are not sent to Wikipedia retrieval.

They are labeled:

`Retrieval_Status = "SKIPPED_EMPTY_CLAIM"`

This preserves claim-extraction failures for later end-to-end evaluation and ablation/error analysis instead of silently deleting them.


In [ ]:
if os.path.exists(OUTPUT_CSV):
    work_df = pd.read_csv(OUTPUT_CSV)
    print("Resuming existing output.")

    # Backward compatibility with output files created before Retrieval_Status existed.
    if "Retrieval_Status" not in work_df.columns:
        work_df["Retrieval_Status"] = ""
else:
    # Remove .head(20) in the loading cell when you are ready to run the full dataset.
    work_df = df.copy()
    work_df["Search_Query"] = ""
    work_df["Wikipedia_Pages"] = ""
    work_df["Retrieval_Status"] = ""

    for rank in range(1, TOP_SENTENCES + 1):
        work_df[f"Evidence_{rank}"] = ""
        work_df[f"Evidence_{rank}_Score"] = None
        work_df[f"Evidence_{rank}_Page"] = ""

for idx in tqdm(range(len(work_df))):
    # ------------------------------------------------------------
    # 1. Skip rows where claim extraction failed / claim is empty.
    # ------------------------------------------------------------
    raw_claim = work_df.at[idx, "Atomic_Claim"]

    if pd.isna(raw_claim) or not str(raw_claim).strip():
        work_df.at[idx, "Search_Query"] = ""
        work_df.at[idx, "Wikipedia_Pages"] = ""
        work_df.at[idx, "Retrieval_Status"] = "SKIPPED_EMPTY_CLAIM"

        for rank in range(1, TOP_SENTENCES + 1):
            work_df.at[idx, f"Evidence_{rank}"] = ""
            work_df.at[idx, f"Evidence_{rank}_Score"] = None
            work_df.at[idx, f"Evidence_{rank}_Page"] = ""

        continue

    claim = str(raw_claim).strip()

    # ------------------------------------------------------------
    # 2. Resume logic: skip rows already successfully retrieved.
    # ------------------------------------------------------------
    existing_status = str(work_df.at[idx, "Retrieval_Status"]).strip()
    existing_evidence = work_df.at[idx, "Evidence_1"]

    if existing_status == "SUCCESS":
        continue

    # Compatibility with an older output that has evidence but no status.
    if (
        not existing_status
        and isinstance(existing_evidence, str)
        and existing_evidence.strip()
        and not existing_evidence.startswith("ERROR:")
    ):
        work_df.at[idx, "Retrieval_Status"] = "SUCCESS"
        continue

    question = str(work_df.at[idx, "Question"])

    try:
        result = retrieve_evidence_for_claim(question, claim)

        work_df.at[idx, "Search_Query"] = result["query"]
        work_df.at[idx, "Wikipedia_Pages"] = json.dumps(
            result["pages"],
            ensure_ascii=False
        )

        # Clear old evidence in case this row is being retried.
        for rank in range(1, TOP_SENTENCES + 1):
            work_df.at[idx, f"Evidence_{rank}"] = ""
            work_df.at[idx, f"Evidence_{rank}_Score"] = None
            work_df.at[idx, f"Evidence_{rank}_Page"] = ""

        for rank in range(1, TOP_SENTENCES + 1):
            if rank <= len(result["evidence"]):
                ev = result["evidence"][rank - 1]
                work_df.at[idx, f"Evidence_{rank}"] = ev["context"]
                work_df.at[idx, f"Evidence_{rank}_Score"] = ev["similarity"]
                work_df.at[idx, f"Evidence_{rank}_Page"] = ev["page_title"]

        work_df.at[idx, "Retrieval_Status"] = "SUCCESS"

    except Exception as e:
        print(f"Row {idx} failed: {e}")
        work_df.at[idx, "Evidence_1"] = f"ERROR: {e}"
        work_df.at[idx, "Retrieval_Status"] = "ERROR"

    # Save frequently so an interruption does not lose much work.
    if (idx + 1) % 10 == 0:
        work_df.to_csv(OUTPUT_CSV, index=False)

work_df.to_csv(OUTPUT_CSV, index=False)

# Explicit final cache save.
_save_json_cache(PAGE_CACHE, PAGE_CACHE_PATH)
_save_json_cache(SEARCH_CACHE, SEARCH_CACHE_PATH)

print("Saved:", OUTPUT_CSV)
print("Page cache entries:", len(PAGE_CACHE))
print("Search cache entries:", len(SEARCH_CACHE))

print("\nRetrieval status:")
print(work_df["Retrieval_Status"].value_counts(dropna=False))


## 9. Inspect retrieval quality

In [ ]:
display_cols = [
    "Question_ID",
    "Claim_ID",
    "Atomic_Claim",
    "Retrieval_Status",
    "Search_Query",
    "Evidence_1_Page",
    "Evidence_1_Score",
    "Evidence_1"
]
work_df[display_cols].head()


In [ ]:
score_col = pd.to_numeric(work_df["Evidence_1_Score"], errors="coerce")

print("Rows:", len(work_df))
print("\nRetrieval status counts:")
print(work_df["Retrieval_Status"].value_counts(dropna=False))

print("\nMean Top-1 similarity:", score_col.mean())
print("Median Top-1 similarity:", score_col.median())
score_col.describe()
